# Addition of translation speed and azimuth to Extended IBTrACS

In [1]:
# Setup environment
import huracanpy
import xarray as xr
import numpy as np

from tqdm import tqdm

In [2]:
# Script parameters
## Extended IBTrACS file you want to add the attributes to (may be the minimal one or one you already added info to)
IN_FILE = "../files/extended_ibtracs.nc"
## Output file automatically determined. You may change if you like.
OUT_FILE = IN_FILE[:-3]+"_distaz"+".nc"

In [3]:
# Open extended IBTrACS
eib = xr.open_dataset(IN_FILE,)
eib["record"] = eib.record # Explicit record as a coordinate

In [4]:
# Parameters computation
S = []
A = []
for ds in tqdm(eib.dataset):
    eib_ds = eib.sel(dataset = ds) # Subset dataset
    eib_ds = eib_ds.where(~np.isnan(eib_ds.lon), drop = True) # Remove points where given dataset does not have data
    speed = huracanpy.calc.translation_speed(eib_ds.lon, eib_ds.lat, eib_ds.time, eib_ds.track_id)
    S.append(speed)
    azimuth = huracanpy.calc.azimuth(eib_ds.lon, eib_ds.lat, eib_ds.track_id)
    A.append(azimuth)
eib["translation_speed"] = xr.concat(S, dim = "dataset")
eib["azimuth"] = xr.concat(A, dim = "dataset")

100%|████████████████████████████████████████████████████████████████████| 6/6 [00:26<00:00,  4.36s/it]


In [5]:
# Visualise new catalogue
eib

<xarray.Dataset> Size: 77MB
Dimensions:            (record: 188964, dataset: 6)
Coordinates:
  * dataset            (dataset) <U17 408B 'IBTrACS' ... 'TRACK-ECMWF-OP-AN'
  * record             (record) int64 2MB 0 1 2 3 ... 188961 188962 188963
Data variables:
    track_id           (record) <U13 10MB ...
    lon                (dataset, record) float64 9MB ...
    lat                (dataset, record) float64 9MB ...
    name               (record) <U13 10MB ...
    pres               (dataset, record) float64 9MB ...
    wind10             (dataset, record) float64 9MB ...
    time               (record) datetime64[ns] 2MB ...
    translation_speed  (dataset, record) float64 9MB nan nan nan ... nan nan nan
    azimuth            (dataset, record) float64 9MB nan nan nan ... nan nan nan

In [6]:
# Save
eib.to_netcdf(OUT_FILE)